## 测试 jax.lax.div

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import jax
import jax.numpy as jnp
import torch
import numpy as np

rng = np.random.default_rng(seed=0)

In [ ]:
def to_numpy_f32(arr):
    if isinstance(arr, torch.Tensor):
        arr = arr.detach().cpu().numpy()
    else:
        arr = np.asarray(arr)
    return np.ascontiguousarray(arr.astype(np.float32, copy=False))
def ulp_diff_float32(a, b):
    a_i = a.view(np.int32).astype(np.int64)
    b_i = b.view(np.int32).astype(np.int64)
    a_ordered = np.where(a_i < 0, 0x80000000 - a_i, a_i)
    b_ordered = np.where(b_i < 0, 0x80000000 - b_i, b_i)
    return np.abs(a_ordered - b_ordered)
def compare_metrics(name, test, ref):
    test = to_numpy_f32(test)
    ref = to_numpy_f32(ref)

    finite_mask = np.isfinite(test) & np.isfinite(ref)
    skipped = test.size - int(finite_mask.sum())
    test_f = test[finite_mask]
    ref_f = ref[finite_mask]

    abs_err = np.abs(test_f - ref_f)
    rel_denom = np.maximum(np.abs(ref_f), np.finfo(np.float32).eps)
    rel_err = abs_err / rel_denom
    ulp_err = ulp_diff_float32(test_f, ref_f)

    print(f"\n{name}")
    print(
        f"  valid elements: {test_f.size}/{test.size}, skipped non-finite pairs: {skipped}"
    )
    print(
        f"  abs_err: max={abs_err.max():.6e}, mean={abs_err.mean():.6e}, p99={np.percentile(abs_err, 99):.6e}"
    )
    print(
        f"  rel_err: max={rel_err.max():.6e}, mean={rel_err.mean():.6e}, p99={np.percentile(rel_err, 99):.6e}"
    )
    print(
        f"  ulp_err: max={int(ulp_err.max())}, mean={ulp_err.mean():.6f}, p99={np.percentile(ulp_err, 99):.6f}"
    )
def find_max_error_index(o_jax, o_torch):
    abs_diff = np.abs(to_numpy_f32(o_jax) - to_numpy_f32(o_torch))

    # 找到全局误差最大的索引位置
    max_diff_idx = np.unravel_index(np.argmax(abs_diff), abs_diff.shape)

    print(f"最大误差发生处的索引: {max_diff_idx}")
    print(f"JAX输入 x: {o_jax[max_diff_idx]}, y: {o_jax[max_diff_idx]}")
    print(f"JAX输出: {o_jax[max_diff_idx]}")
    print(f"Torch输出: {o_torch[max_diff_idx]}")

In [3]:
x = rng.normal(size=(1024,1024))
y = rng.normal(size=(1024,1024))

In [4]:
x_jax_32 = jnp.array(x)
y_jax_32 = jnp.array(y)
x_torch_32 = torch.from_numpy(x).to(torch.float32).cuda()
y_torch_32 = torch.from_numpy(y).to(torch.float32).cuda()

In [5]:
o_jax_32 = jax.lax.div(x_jax_32, y_jax_32)
o_torch_32 = torch.ops.aten.div.Tensor(x_torch_32, y_torch_32)
compare_metrics("fp32: jax vs torch", o_jax_32, o_torch_32)
find_max_error_index(o_jax_32, o_torch_32)


fp32: jax vs torch
  valid elements: 1048576/1048576, skipped non-finite pairs: 0
  abs_err: max=7.812500e-03, mean=1.799598e-07, p99=1.907349e-06
  rel_err: max=1.722126e-07, mean=2.373307e-08, p99=1.153927e-07
  ulp_err: max=2, mean=0.287199, p99=1.000000
最大误差发生处的索引: (np.int64(801), np.int64(792))
JAX输入 x: -67941.0703125, y: -67941.0703125
JAX输出: -67941.0703125
Torch输出: -67941.0625


---

根据如上结果可以看出，存在较大误差，虽然 ulp 只有 1～2个，但绝对误差竟然达到了 1e-3，这很可怕。
于是下面进行探究是哪里出了问题，首先，看看是否是用了jax 或者 torch 是否用了快速/近似除法路径

---

In [6]:
import torch._dynamo
from torch._inductor import config as inductor_config

torch._dynamo.reset()
inductor_config.emulate_divison_rounding = True
@torch.compile
def compiled_div(a, b):
    return torch.ops.aten.div.Tensor(a, b)

o_torch_32_compiled = compiled_div(x_torch_32, y_torch_32)
compare_metrics("fp32: jax vs torch compiled(inductor_config.emulate_divison_rounding = True)", o_jax_32, o_torch_32_compiled)
find_max_error_index(o_jax_32, o_torch_32_compiled)


fp32: jax vs torch compiled(inductor_config.emulate_divison_rounding = True)
  valid elements: 1048576/1048576, skipped non-finite pairs: 0
  abs_err: max=7.812500e-03, mean=1.799598e-07, p99=1.907349e-06
  rel_err: max=1.722126e-07, mean=2.373307e-08, p99=1.153927e-07
  ulp_err: max=2, mean=0.287199, p99=1.000000
最大误差发生处的索引: (np.int64(801), np.int64(792))
JAX输入 x: -67941.0703125, y: -67941.0703125
JAX输出: -67941.0703125
Torch输出: -67941.0625


In [7]:
inductor_config.emulate_divison_rounding = False
@torch.compile
def compiled_div(a, b):
    return torch.ops.aten.div.Tensor(a, b)

o_torch_32_compiled = compiled_div(x_torch_32, y_torch_32)
compare_metrics("fp32: jax vs torch compiled(inductor_config.emulate_divison_rounding = False)", o_jax_32, o_torch_32_compiled)
find_max_error_index(o_jax_32, o_torch_32_compiled)


fp32: jax vs torch compiled(inductor_config.emulate_divison_rounding = False)
  valid elements: 1048576/1048576, skipped non-finite pairs: 0
  abs_err: max=0.000000e+00, mean=0.000000e+00, p99=0.000000e+00
  rel_err: max=0.000000e+00, mean=0.000000e+00, p99=0.000000e+00
  ulp_err: max=0, mean=0.000000, p99=0.000000
最大误差发生处的索引: (np.int64(0), np.int64(0))
JAX输入 x: 0.10312499850988388, y: 0.10312499850988388
JAX输出: 0.10312499850988388
Torch输出: 0.10312499850988388


---

分析：根据如上设置，可以发现 当使用 Inductor 编译器编译时，如果设置 inductor_config.emulate_divison_rounding = False，与 jax 一致，如果设为 True 则与torch的 eager 模式有一样的误差。

差异主要来自除法舍入语义/实现路径

下面继续探索编辑器层面的

---

In [8]:
# JAX
import os
from pathlib import Path

DUMP_DIR = Path("./xla_div_dump")
DUMP_DIR.mkdir(parents=True, exist_ok=True)

# 必须在 import jax 之前设置
os.environ["XLA_FLAGS"] = f"--xla_dump_to={DUMP_DIR}"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import jax
import jax.numpy as jnp

def f(x, y):
    return jax.lax.div(x, y)

x = jnp.ones((1024, 1024), dtype=jnp.float32)
y = jnp.ones((1024, 1024), dtype=jnp.float32)

lowered = jax.jit(f).lower(x, y)

# 1) StableHLO
(DUMP_DIR / "stablehlo.mlir").write_text(str(lowered.compiler_ir(dialect="stablehlo")))

# 2) HLO
hlo = lowered.compiler_ir(dialect="hlo")
(DUMP_DIR / "hlo.txt").write_text(hlo.as_hlo_text())

# 3/4) LLVM/PTX: 触发真正编译后会落在 xla_dump_to 目录
compiled = lowered.compile()
_ = compiled(x, y).block_until_ready()

---

前端 IR 是“纯 divide”，没有被改写。

  - HLO：hlo.txt:6 (/home/gcpuser/sky_workdir/FUEL-JAX/op_test/div/xla_div_dump/hlo.txt:6) 是 ROOT ... divide(x.1, y.1)
  - StableHLO：stablehlo.mlir:3 (/home/gcpuser/sky_workdir/FUEL-JAX/op_test/div/xla_div_dump/stablehlo.mlir:3) 是 stablehlo.divide

  这说明问题不在前端（StableHLO/HLO）层，而在更后面的 GPU lowering/codegen（LLVM/PTX）层。

---

---

```bash
XLA_FLAGS="--xla_dump_to=/tmp/xla_div_backend --xla_dump_hlo_pass_re=.*" \
  /home/gcpuser/sky_workdir/FUEL-JAX/.venv/bin/python - <<'PY'
  import jax, jax.numpy as jnp
  def f(x, y): return jax.lax.div(x, y)
  x = jnp.ones((1024,1024), jnp.float32)
  y = jnp.ones_like(x)
  c = jax.jit(f).lower(x, y).compile()
  _ = c(x, y).block_until_ready()
  print("done")
  PY
```

这个 ptx 很“干净”，结论是它走的是标准除法路径，不是近似倒数。

  关键点：

  1. Kernel 基本信息

  - 目标架构：sm_90a（H100），见 module_1061.jit_f.ptx:6 (/tmp/xla_div_backend/module_1061.jit_f.ptx:6)
  - block 线程数：128，见 module_1061.jit_f.ptx:16 (/tmp/xla_div_backend/module_1061.jit_f.ptx:16)

  2. 计算映射（每线程处理4个元素）

  - 索引构造：tid<<2 和 ctaid<<9 组合，见 module_1061.jit_f.ptx:29 (/tmp/xla_div_backend/module_1061.jit_f.ptx:29)、module_1061.jit_f.ptx:30 (/tmp/xla_div_backend/module_1061.jit_f.ptx:30)、
    module_1061.jit_f.ptx:31 (/tmp/xla_div_backend/module_1061.jit_f.ptx:31)
  - 等价于：elem_idx = blockIdx.x*512 + threadIdx.x*4

  3. 指令层核心

  - 读取是 ld.global.nc.v4.b32，一次读4个 f32，见 module_1061.jit_f.ptx:35 (/tmp/xla_div_backend/module_1061.jit_f.ptx:35)、module_1061.jit_f.ptx:36 (/tmp/xla_div_backend/module_1061.jit_f.ptx:36)
  - 计算是 div.full.f32，见 module_1061.jit_f.ptx:37 (/tmp/xla_div_backend/module_1061.jit_f.ptx:37)-40 (/tmp/xla_div_backend/module_1061.jit_f.ptx:40)
  - 写回是 st.global.v4.b32，见 module_1061.jit_f.ptx:42 (/tmp/xla_div_backend/module_1061.jit_f.ptx:42)

  4. 你最关心的“是否近似除法”

  - 这个文件里没有 rcp.approx.f32，也没有 mul.rn.f32 的 reciprocal-mul 组合。
  - LLVM 层也是直接 fdiv，见 module_1061.jit_f.ir-with-opt.ll:29 (/tmp/xla_div_backend/module_1061.jit_f.ir-with-opt.ll:29)-32 (/tmp/xla_div_backend/module_1061.jit_f.ir-with-opt.ll:32)。
  - HLO 源头就是 divide(x, y)，见 module_1061.jit_f.before_optimizations.txt:6 (/tmp/xla_div_backend/module_1061.jit_f.before_optimizations.txt:6)。

  一句话：这个 JAX kernel 本身是“直接除法”实现，不是近似倒数实现。
  如果还要追 bug，下一步应对比 Torch 对应 kernel 的 PTX/SASS，确认它是不是另一条除法路径。

  ---

---

```bash
TORCH_LOGS=output_code TORCHINDUCTOR_CACHE_DIR=/tmp/ti_div_false \
/home/gcpuser/sky_workdir/FUEL-JAX/.venv/bin/python - <<'PY'
import torch
from torch._inductor import config
config.emulate_divison_rounding = False

@torch.compile(fullgraph=True)
def f(a, b):
    return torch.ops.aten.div.Tensor(a, b)

a = torch.randn(1024, 1024, device="cuda", dtype=torch.float32)
b = torch.randn(1024, 1024, device="cuda", dtype=torch.float32)
_ = f(a, b); torch.cuda.synchronize()
print("done false")
PY
```

1. Torch compile + emulate_divison_rounding=False
- Triton 源是普通 /：cnp57ant...py:26 (/tmp/ti_div_false/np/cnp57ant5ultng3cloj6okrygxrwxepxiju35nx4uy3mwl5xjxuh.py:26)
- 落到 PTX 是 div.full.f32：triton_poi_fused_div_0.ptx:80 (/tmp/ti_div_false/triton/0/A5VH47JGRNDPIBEYTQKNG3EQZ2VYUYJ7SBE3BUF3IFDG7GRSX4UQ/triton_poi_fused_div_0.ptx:80)

2. Torch compile + emulate_divison_rounding=True
- Triton 源变成 triton.language.div_rn(...)：cgssj3x...py:26 (/tmp/ti_div_true/gs/cgssj3xa5wjoobmttrqpb7td4nfnxqf6xysbjjyamw3of4qcn6yu.py:26)
- PTX 是 div.rn.f32：triton_poi_fused_div_0.ptx:60 (/tmp/ti_div_true/triton/0/OSLCEWP6EKKWRIAG5V7XFYLSFSPIKYS32XLNBROMCFDY7TPXAI6A/triton_poi_fused_div_0.ptx:60)

3. 数值也完全对应
- jax vs torch compile false：全 0
- jax vs torch compile true：abs_max=7.8125e-03, ulp_max=2
- torch eager vs compile true：全 0

---

---

- div.rn.f32：标准IEEE 754单精度除法，采用“向最近偶数舍入”（round-to-nearest-even）模式，结果严格正确舍入，精度最高，通常较慢。
- div.full.f32：全区间浮点除法，支持所有指数范围和特殊值，通常基于倒数近似加牛顿迭代实现，精度接近但不保证严格舍入，误差可达约2个ULP，速度常优于div.rn.f32。
- div.approx.f32：快速近似除法，误差较大，性能最高，可能忽略非正规数。

观察到div.full.f32与div.rn.f32结果有小的ULP差异，说明div.rn.f32严格保证舍入，div.full.f32则为性能做了折中。

总结：
| 指令         | 精度                      | 舍入方式                   | 性能             | 适用情况                 |
|--------------|-------------------------|-------------------------|----------------|----------------------|
| div.rn.f32   | 严格舍入单精度除法           | 向最近偶数舍入（IEEE 754标准）    | 较慢             | 需严格正确除法结果        |
| div.full.f32 | 全指数范围，接近正确舍入    | 不保证严格舍入，通常最接近round-to-nearest | 较快             | 需全范围支持和较好性能时   |
| div.approx.f32 | 快速近似，误差较大            | 无严格保证                 | 快速             | 性能优先，允许较大误差时     |

---

最终结论

jax.lax.div 使用的是 div.full.f32，不保证严格舍入，通常最接近round-to-nearest

torch.ops.aten.div.Tensor 使用的是 div.rn.f32，标准IEEE 754单精度除法，通常最接近round-to-nearest-even